# Quantized LLM Evaluation on Colab
## 4-Model Core Set × 4 Quantization Precisions

**Available Precisions:**
- `int8` — 8-bit integer (universal support)
- `int4` — 4-bit integer (TensorRT, ONNX, broader edge device support)
- `nf4` — 4-bit Normal Float (NVIDIA GPUs only)
- `fp8` — 8-bit floating-point (modern GPUs + emerging edge support)

**GPU / TPU Compatibility:**
- ✅ NF4 supported: H100, G4, A100, L4, T4 GPUs
- ❌ NF4 NOT supported: v6e-1 TPU, v5e-1 TPU
- ⚠️ Edge devices (RPi5, Jetson Xavier/Orin, Qualcomm): Limited/no standard NF4 support

**Recommended Setup:** L4 GPU on Colab

In [ ]:
import os
import platform
import subprocess
import sys

import torch

print("=" * 70)
print("ENVIRONMENT & GPU DIAGNOSTICS")
print("=" * 70)
print(f"Platform: {platform.platform()}")
print(f"Python: {platform.python_version()}")
print(f"PyTorch: {torch.__version__}")
print()

print("CUDA / GPU:")
print(f"  torch.cuda.is_available(): {torch.cuda.is_available()}")
print(f"  torch.cuda.device_count(): {torch.cuda.device_count()}")
if torch.cuda.is_available():
    idx = torch.cuda.current_device()
    props = torch.cuda.get_device_properties(idx)
    print(f"  Current device index: {idx}")
    print(f"  GPU name: {torch.cuda.get_device_name(idx)}")
    print(f"  Compute capability: {props.major}.{props.minor}")
    print(f"  Total VRAM (GiB): {props.total_memory / (1024**3):.2f}")
else:
    print("  No CUDA GPU detected by PyTorch.")
print()

print("nvidia-smi:")
try:
    out = subprocess.check_output(["nvidia-smi"], stderr=subprocess.STDOUT, text=True)
    print(out)
except Exception as e:
    print(f"  nvidia-smi unavailable: {e}")

print("=" * 70)

## Step 1: Clone Repository & Install Dependencies

In [ ]:
import os
import subprocess
import sys

repo_dir = "/content/intent-classifier"
if not os.path.isdir(repo_dir):
    print("Cloning intent-classifier repository...")
    subprocess.run(
        ["git", "clone", "https://github.com/kon172verma/intent-classifier.git", repo_dir],
        check=True,
    )
    print("Repository cloned successfully.\n")
else:
    print(f"Repository already present at {repo_dir}\n")

print("Installing Python dependencies...")
subprocess.run(
    [
        "pip", "install", "-q",
        "transformers",
        "accelerate",
        "bitsandbytes",
        "torch",
        "python-dotenv",
        "matplotlib",
        "numpy",
    ],
    check=True,
)
print("Dependencies installed.\n")

os.chdir(repo_dir)
print(f"Working directory: {os.getcwd()}\n")

## Step 2: Set Up Hugging Face Token

In [ ]:
import os
from dotenv import load_dotenv
from huggingface_hub import login

hf_token = None
token_source = None

try:
    from google.colab import userdata  # type: ignore
    secret_token = userdata.get("HF_TOKEN")
    if secret_token:
        hf_token = secret_token
        token_source = "Colab Secret (HF_TOKEN)"
except Exception:
    pass

if not hf_token:
    env_token = os.getenv("HF_TOKEN")
    if env_token:
        hf_token = env_token
        token_source = "Environment variable (HF_TOKEN)"

if not hf_token:
    env_file = "/content/intent-classifier/.env"
    if os.path.exists(env_file):
        load_dotenv(env_file, override=False)
        dotenv_token = os.getenv("HF_TOKEN")
        if dotenv_token:
            hf_token = dotenv_token
            token_source = ".env file (/content/intent-classifier/.env)"

print("HF auth token source check complete.")
if hf_token:
    print(f"Using token from: {token_source}")
    login(token=hf_token, add_to_git_credential=False)
    print("Hugging Face authentication successful.\n")
else:
    print("HF_TOKEN not found — gated models will be skipped.\n")

## Step 3: Configuration & Model Definitions

Single source of truth for models (4-core set) and quantization precisions.
Base model IDs are used (not instruct variants) for cleaner NF4 quantization.

In [ ]:
import sys

# evaluation_lib lives at repo root — must be on sys.path
sys.path.insert(0, "/content/intent-classifier")

# ── 4-MODEL CORE SET (base model IDs) ──────────────────────────────────────
SELECTED_MODELS = {
    "qwen2.5-0.5b": "Qwen/Qwen2.5-0.5B",       # base (was Instruct)
    "qwen3-0.6b":   "Qwen/Qwen3-0.6B",          # unified base+chat
    "qwen2.5-1.5b": "Qwen/Qwen2.5-1.5B",       # base (was Instruct)
    "smollm3":      "HuggingFaceTB/SmolLM3-3B", # no separate base
}

MODEL_DISPLAY = {
    "qwen2.5-0.5b": "Qwen2.5-0.5B (~500M)",
    "qwen3-0.6b":   "Qwen3-0.6B (~600M)",
    "qwen2.5-1.5b": "Qwen2.5-1.5B (~1.5B)",
    "smollm3":      "SmolLM3-3B (~3.0B)",
}

# ── 4 QUANTIZATION PRECISIONS ──────────────────────────────────────────────
QUANTIZATION_PRECISIONS = {
    "int8": "8-bit integer (universal)",
    "int4": "4-bit integer (TensorRT, ONNX compatible)",
    "nf4":  "4-bit Normal Float (NVIDIA GPUs only)",
    "fp8":  "8-bit float (modern GPUs + edge)",
}

# ── DEFAULT PATHS ──────────────────────────────────────────────────────────
DEFAULT_DATA_PATH  = "/content/intent-classifier/dataset_sample/sample.json"
QUANTIZED_EVAL_DIR = "/content/intent-classifier/evaluation_quantized"

print("=" * 70)
print("CONFIGURATION: 4 Models × 4 Quantization Precisions")
print("=" * 70)
print("\nModels (base model IDs):")
for key, mid in SELECTED_MODELS.items():
    print(f"  - {MODEL_DISPLAY[key]}  →  {mid}")

print("\nQuantization Precisions:")
for prec, desc in QUANTIZATION_PRECISIONS.items():
    print(f"  - {prec:<8} : {desc}")

print(f"\nData path: {DEFAULT_DATA_PATH}")
print(f"Output dir: {QUANTIZED_EVAL_DIR}")
print("=" * 70 + "\n")

## Step 4: NF4 Compatibility & Hardware Support Matrix

In [ ]:
print("=" * 80)
print("NF4 QUANTIZATION HARDWARE SUPPORT MATRIX")
print("=" * 80)
print()

print("DATA CENTER & CLOUD GPUs (NF4 Support via CUDA + BitsAndBytes):")
for gpu in ["H100 GPU", "G4 GPU", "A100 GPU", "L4 GPU", "T4 GPU"]:
    print(f"  ✅ {gpu:<20} Full support for NF4 quantization")
print()

print("TPUs (NF4 Support Status):")
for tpu in ["v6e-1 TPU", "v5e-1 TPU"]:
    print(f"  ❌ {tpu:<20} No native NF4 support (uses bfloat16/int8 instead)")
print()

print("EDGE DEVICES (NF4 Support Status):")
print("  ⚠️  Raspberry Pi 5      NOT supported (no NVIDIA BitsAndBytes)")
print("  ⚠️  NVIDIA Jetson Xavier Limited (TensorRT support is partial)")
print("  ⚠️  NVIDIA Jetson Orin  Limited (TensorRT support is partial)")
print("  ❌ Qualcomm devices    NOT supported (QNN uses proprietary format)")
print()

print("RECOMMENDATION FOR COLAB:")
print("  This notebook runs on cloud GPUs (H100, A100, L4, T4).")
print("  NF4 quantization is fully supported on these platforms.")
print()
print("=" * 80)

## Step 5: Run Quantization Evaluation (Smoke Test)

Quick test: run the smallest model (`qwen2.5-0.5b`) across all 4 quantization precisions on a subset of data (10 examples).

In [ ]:
import subprocess
import sys
from pathlib import Path
import time
import torch

SMOKE_TEST_MODEL  = "qwen2.5-0.5b"
SMOKE_TEST_QUANTS = ["fp8", "nf4", "int4", "int8"]
SMOKE_TEST_LIMIT  = 10

eval_script = Path(QUANTIZED_EVAL_DIR) / "src" / "quant_eval.py"

def get_gpu_info():
    if torch.cuda.is_available():
        i = torch.cuda.current_device()
        return i, torch.cuda.get_device_name(i)
    return None, "CPU"

print("=" * 90)
print("QUANTIZATION EVALUATION SMOKE TEST")
print("=" * 90)
print(f"Model   : {SELECTED_MODELS.get(SMOKE_TEST_MODEL)}")
print(f"Quants  : {', '.join(SMOKE_TEST_QUANTS)}")
print(f"Examples: {SMOKE_TEST_LIMIT}")
print("=" * 90)
print()

for idx, quant in enumerate(SMOKE_TEST_QUANTS, 1):
    dev_idx, dev_name = get_gpu_info()
    print(f"[{idx}/{len(SMOKE_TEST_QUANTS)}] {SMOKE_TEST_MODEL} [{quant.upper()}]  GPU: {dev_name}")

    start = time.time()
    cmd = [
        sys.executable, str(eval_script),
        "--model",  SMOKE_TEST_MODEL,
        "--quant",  quant,
        "--mode",   "zero_shot",
        "--limit",  str(SMOKE_TEST_LIMIT),
        "--device", "auto",
        "--data",   DEFAULT_DATA_PATH,
    ]
    result = subprocess.run(cmd, text=True, capture_output=True)
    elapsed = time.time() - start

    if result.returncode == 0:
        print(f"  SUCCESS | {elapsed:.1f}s")
        if result.stdout.strip():
            print(result.stdout)
    else:
        print(f"  FAILED  | exit={result.returncode} | {elapsed:.1f}s")
        if result.stdout.strip():
            print(result.stdout)
        if result.stderr.strip():
            print(result.stderr)
    print("-" * 90)

print()
print("=" * 90)
print("SMOKE TEST SUMMARY")
print("=" * 90)
print(f"Model evaluated: {SELECTED_MODELS.get(SMOKE_TEST_MODEL)}")
print(f"Precisions tested: {', '.join(SMOKE_TEST_QUANTS)}")
print("Results saved to:")
for quant in SMOKE_TEST_QUANTS:
    print(f"  - {QUANTIZED_EVAL_DIR}/reports_{quant}/")
print("=" * 90)

## Step 6: Full Zero-Shot Evaluation (All Models × All Precisions)

Runs batch evaluation on `sample.json` across all 4 models and all 4 precisions in zero-shot mode.
This clears quantized report folders first so smoke-test artifacts are not mixed into full-run results.

In [ ]:
import subprocess
import sys
from pathlib import Path
import shutil

print("=" * 90)
print("FULL ZERO-SHOT EVALUATION | ALL MODELS x ALL PRECISIONS | sample.json")
print("=" * 90)

quant_eval_dir = Path(QUANTIZED_EVAL_DIR)
runner_script  = quant_eval_dir / "src" / "run_quant_eval.py"
sample_data    = Path("/content/intent-classifier/dataset_sample/sample.json")

# Clear smoke-test artifacts
for d in ["reports_int8", "reports_int4", "reports_nf4", "reports_fp8"]:
    p = quant_eval_dir / d
    if p.exists():
        shutil.rmtree(p)
    p.mkdir(parents=True, exist_ok=True)

cmd = [
    sys.executable, str(runner_script),
    "--mode",   "zero_shot",
    "--data",   str(sample_data),
    "--device", "auto",
]

print("Running command:")
print(" ".join(cmd))
print()

result = subprocess.run(cmd, cwd=str(quant_eval_dir), text=True)

if result.returncode == 0:
    print("\nZERO-SHOT FULL SWEEP COMPLETED SUCCESSFULLY")
else:
    print(f"\nZERO-SHOT FULL SWEEP FAILED (exit={result.returncode})")

print("=" * 90)

## Step 7: Full Few-Shot Evaluation (All Models × All Precisions)

In [ ]:
import subprocess
import sys
from pathlib import Path

print("=" * 90)
print("FULL FEW-SHOT EVALUATION | ALL MODELS x ALL PRECISIONS | sample.json")
print("=" * 90)

quant_eval_dir = Path(QUANTIZED_EVAL_DIR)
runner_script  = quant_eval_dir / "src" / "run_quant_eval.py"
sample_data    = Path("/content/intent-classifier/dataset_sample/sample.json")

cmd = [
    sys.executable, str(runner_script),
    "--mode",   "few_shot",
    "--data",   str(sample_data),
    "--device", "auto",
]

print("Running command:")
print(" ".join(cmd))
print()

result = subprocess.run(cmd, cwd=str(quant_eval_dir), text=True)

if result.returncode == 0:
    print("\nFEW-SHOT FULL SWEEP COMPLETED SUCCESSFULLY")
else:
    print(f"\nFEW-SHOT FULL SWEEP FAILED (exit={result.returncode})")

print("=" * 90)

## Step 8: Validate Coverage & Generate Comparison Plots

In [ ]:
import json
import subprocess
import sys
from pathlib import Path
import time

plot_script = Path(QUANTIZED_EVAL_DIR) / "src" / "plot_quant.py"
report_dirs = [
    Path(QUANTIZED_EVAL_DIR) / "reports_int8",
    Path(QUANTIZED_EVAL_DIR) / "reports_int4",
    Path(QUANTIZED_EVAL_DIR) / "reports_nf4",
    Path(QUANTIZED_EVAL_DIR) / "reports_fp8",
]

expected_models = set(SELECTED_MODELS.keys())
expected_quants = {"int8", "int4", "nf4", "fp8"}
expected_modes  = {"zero_shot", "few_shot"}
expected_combos = {
    (m, q, mode)
    for m in expected_models
    for q in expected_quants
    for mode in expected_modes
}

sample_data        = Path("/content/intent-classifier/dataset_sample/sample.json")
expected_n_examples = len(json.loads(sample_data.read_text(encoding="utf-8")))

reports = []
for d in report_dirs:
    if d.exists():
        for f in d.glob("*.json"):
            try:
                reports.append(json.loads(f.read_text(encoding="utf-8")))
            except Exception:
                pass

seen_combos    = {(r.get("model_key"), r.get("quant"), r.get("eval_mode")) for r in reports}
missing        = sorted(expected_combos - seen_combos)
invalid_sizes  = [
    (r.get("model_key"), r.get("quant"), r.get("eval_mode"), r.get("n_examples"))
    for r in reports if r.get("n_examples") != expected_n_examples
]

print("=" * 90)
print("FULL-COVERAGE VALIDATION BEFORE PLOTTING")
print("=" * 90)
print(f"Expected sample size per run: {expected_n_examples}")
print(f"Reports found: {len(reports)}  /  Expected: {len(expected_combos)}")

if missing:
    print("\nMissing combinations:")
    for item in missing[:20]:
        print(" ", item)
    raise RuntimeError("Cannot plot: full coverage is incomplete.")

if invalid_sizes:
    print("\nReports not from full sample.json:")
    for item in invalid_sizes[:20]:
        print(" ", item)
    raise RuntimeError("Cannot plot: some reports are from partial runs.")

print("Coverage check passed.")
print()

start = time.time()
cmd   = [
    sys.executable, str(plot_script),
    "--out-dir", str(Path(QUANTIZED_EVAL_DIR) / "analysis"),
]
result  = subprocess.run(cmd, cwd=QUANTIZED_EVAL_DIR, capture_output=False)
elapsed = time.time() - start

if result.returncode == 0:
    print(f"PLOT GENERATION COMPLETED | {elapsed:.1f}s")
    plot_dir = Path(QUANTIZED_EVAL_DIR) / "analysis"
    for img in sorted(plot_dir.glob("*.png")):
        print(f"  - {img.name}")
else:
    raise RuntimeError(f"Plot generation failed (exit {result.returncode})")

print("=" * 90)

## Step 9: Download Results (Optional)

In [ ]:
import shutil
from pathlib import Path

print("=" * 90)
print("PREPARING RESULTS FOR DOWNLOAD")
print("=" * 90)

try:
    from google.colab import files  # type: ignore
    print("\u2713 Detected Google Colab environment \u2014 Download available")

    results_dir = Path("/tmp/quantized_eval_results")
    results_dir.mkdir(exist_ok=True)

    copied_dirs = []
    for report_dir_name in ["reports_int8", "reports_int4", "reports_nf4", "reports_fp8"]:
        src_dir = Path(QUANTIZED_EVAL_DIR) / report_dir_name
        if src_dir.exists() and list(src_dir.glob("*.json")):
            dst = results_dir / report_dir_name
            if dst.exists():
                shutil.rmtree(dst)
            shutil.copytree(src_dir, dst)
            print(f"  \u2713 {report_dir_name}/ ({len(list(dst.glob('*.json')))} reports)")
            copied_dirs.append(report_dir_name)

    analysis_src = Path(QUANTIZED_EVAL_DIR) / "analysis"
    if analysis_src.exists() and list(analysis_src.glob("*.png")):
        analysis_dst = results_dir / "analysis"
        if analysis_dst.exists():
            shutil.rmtree(analysis_dst)
        shutil.copytree(analysis_src, analysis_dst)
        print(f"  \u2713 analysis/ ({len(list(analysis_dst.glob('*.png')))} plots)")
        copied_dirs.append("analysis")

    if copied_dirs:
        archive = shutil.make_archive(
            "/tmp/quantized_eval_results",
            "zip",
            results_dir.parent,
            results_dir.name,
        )
        print(f"\nArchive size: {Path(archive).stat().st_size / (1024*1024):.2f} MB")
        files.download(archive)
        print("\u2705 Download complete!")
    else:
        print("  \u26a0\ufe0f  No results found to download")

except ImportError:
    print("\u2139 Not running on Colab \u2014 Results remain in workspace")
    print("\nResults locations:")
    for quant in ["int8", "int4", "nf4", "fp8"]:
        report_dir = Path(QUANTIZED_EVAL_DIR) / f"reports_{quant}"
        if report_dir.exists():
            print(f"  [{len(list(report_dir.glob('*.json')))} reports] {report_dir}")
    analysis_dir = Path(QUANTIZED_EVAL_DIR) / "analysis"
    if analysis_dir.exists():
        print(f"  [{len(list(analysis_dir.glob('*.png')))} plots]   {analysis_dir}")

print()
print("=" * 90)


## Summary & Next Steps

### ✅ What This Notebook Does
1. **Clones & sets up** the intent-classifier repository in Colab
2. **Installs dependencies** (transformers, torch, bitsandbytes, matplotlib)
3. **Authenticates** with Hugging Face Hub
4. **Runs smoke test** on the smallest model (qwen2.5-0.5b) across all 4 precisions
5. **Runs full evaluation** (all 4 models × 4 precisions × 2 modes)
6. **Generates plots** comparing quantization performance
7. **Downloads results** as a ZIP archive (Colab only)

### 🎯 Configuration
- **Models**: 4-core set with base model IDs (Qwen2.5-0.5B, Qwen3-0.6B, Qwen2.5-1.5B, SmolLM3-3B)
- **Quantizations**: 4 precisions (int8, int4, nf4, fp8)
- **Evaluation metrics**: Accuracy (%), latency (ms), throughput (tokens/s), peak memory (MB)

### 📊 Hardware Support
- **NVIDIA Cloud GPUs** (H100, A100, L4, T4): Full NF4 support ✅
- **TPUs**: Limited (falls back to bfloat16/int8)
- **Edge devices** (RPi5, Jetson, Qualcomm): Partial (int4/fp8 recommended)

### 📚 References
- **Repository**: https://github.com/kon172verma/intent-classifier
- **Shared library**: `evaluation_lib/` (MODEL_REGISTRY, SYSTEM_PROMPTS, eval loop)
- **Evaluation Scripts**: `evaluation_quantized/src/`
  - `quant_eval.py` — Single model evaluation
  - `run_quant_eval.py` — Batch orchestration
  - `plot_quant.py` — Comparative plotting
- **Quantization Library**: [BitsAndBytes](https://github.com/TimDettmers/bitsandbytes)